In [ ]:
import sys, os, glob, shutil
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
if not torch.cuda.is_available(): raise SystemExit("GPU не выделена")
name = torch.cuda.get_device_name(0); print("GPU:", name)
if "T4" not in name and "A100" not in name and "H100" not in name:
    raise SystemExit(f"Нужна T4: {name}")
modules = glob.glob("/kaggle/input/**/train_ce_large.py", recursive=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
for path in glob.glob(os.path.dirname(modules[0]) + "/*.py"):
    shutil.copy(path, "/kaggle/working/src/")
open("/kaggle/working/src/__init__.py", "a").close()
pack = os.path.dirname(glob.glob("/kaggle/input/**/item_texts.parquet", recursive=True)[0])
os.makedirs("/kaggle/working/pack", exist_ok=True)
for path in glob.glob(pack + "/*"):
    dst = "/kaggle/working/pack/" + os.path.basename(path)
    if not os.path.exists(dst): os.symlink(path, dst)
stage1 = glob.glob("/kaggle/input/**/stage1_llm", recursive=True)[0]
print("стартуем с:", stage1)
os.chdir("/kaggle/working"); sys.path.insert(0, "/kaggle/working")
sys.argv = ["train_ce_large", "--prepacked", "/kaggle/working/pack", "--holdout-fold", "0",
            "--base-model", "DeepPavlov/rubert-base-cased", "--resume-from", stage1,
            "--human-relaxed", "--only-categories", "Обувь,Одежда,Ювелирные изделия",
            "--human-epochs", "3", "--batch-size", "256", "--max-length", "256",
            "--output", "/kaggle/working/ce_spec"]
from src.train_ce_large import main
main()
